# Day 06 下午个人项目：电商用户数据可视化
**学号：** 24012434  
**姓名：** 金睿博  
**专题方向：** A（用户生命周期）  
**日期：** 2026年7月  

---

## 项目规则
1. 使用第4天清洗数据，并核对第5天个人分析结果
2. 柱状图和散点图必做；折线图只能用于有序阶段
3. 饼图只用于少量类别的整体构成
4. 每张图写"观察—证据—边界"
5. 输出文件名和目录不得修改

输出目录：`output/day06_visualization/`

---
## 项目配置

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

STUDENT_ID = "24012434"
TOPIC = "A"

pd.set_option("display.max_columns", 50)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# 中文字体设置
plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "PingFang SC", "Arial Unicode MS"]
plt.rcParams["axes.unicode_minus"] = False

# 数据路径
DATA_PATH = Path(r"C:\Users\resch\Downloads\output\day04_project\ecommerce_customer_cleaned.csv")
DAY05_DIR = Path(r"C:\Users\resch\Downloads\output\day05_student\24012434")
OUTPUT_DIR = Path(r"C:\Users\resch\Downloads\output\day06_visualization")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("学生：", STUDENT_ID)
print("专题：", TOPIC)
print("输出：", OUTPUT_DIR)

---
## 检查点1：输入与业务问题

先验证输入文件，再写出4个业务问题和图表选择理由。

In [ ]:
# 验证输入文件
required_inputs = [
    DATA_PATH,
    DAY05_DIR / "overall_metrics.csv",
    DAY05_DIR / "segment_analysis.csv",
    DAY05_DIR / "cross_analysis.csv",
]
for path in required_inputs:
    assert path.exists(), f"缺少文件：{path}"

# 读取数据
df = pd.read_csv(DATA_PATH)
overall_metrics = pd.read_csv(required_inputs[1])
segment_analysis = pd.read_csv(required_inputs[2])
cross_analysis = pd.read_csv(required_inputs[3])

# 验证数据
assert df.shape[0] == 5630, f"数据行数异常：{df.shape[0]}"
assert set(df["Churn"].unique()) == {0, 1}, "Churn取值异常"

print(f"清洗数据：{df.shape}")
print("✅ 检查点1A通过：输入文件有效")

display(overall_metrics)
display(segment_analysis.head())
display(cross_analysis.head())

In [ ]:
# 填写4个业务问题和图表选择理由
business_questions = {
    "category_bar": "不同生命周期阶段的用户流失率是否有显著差异？",
    "behavior_scatter": "用户使用时长与返现金额之间是否存在关联？流失用户分布有何特征？",
    "ordered_line": "随着用户生命周期的增长，流失率和订单数如何变化？",
    "composition_chart": "整体用户中流失与未流失的比例构成是怎样的？",
}

chart_reasons = {
    "category_bar": "TenureGroup是离散类别字段，目标是比较各类别的流失率，柱状图最适合离散类别间的数值对比。",
    "behavior_scatter": "需要展示两个数值变量之间的关系，并用颜色区分离散标签（流失/未流失），散点图最直观。",
    "ordered_line": "TenureGroup具有明确的先后顺序（0-1年→15年），折线图能清晰展示随阶段变化的趋势。",
    "composition_chart": "Churn只有2个类别（流失/未流失），类别少且合计为整体，适合用饼图展示构成。",
}

print("=== 业务问题 ===")
for k, v in business_questions.items():
    print(f"  {k}: {v}")

print("\n=== 图表选择理由 ===")
for k, v in chart_reasons.items():
    print(f"  {k}: {v}")

print("\n✅ 检查点1B通过")

---
## 任务1：类别比较柱状图

**分组字段：** TenureGroup  
**指标：** 用户数 + 流失率  
**问题：** 不同生命周期阶段的用户流失率是否有显著差异？

In [ ]:
# 准备数据
category_field = "TenureGroup"
tenure_order = ["0-1年", "1-5年", "5-10年", "10-15年", "15年以上"]

category_summary = (
    df.groupby(category_field, observed=True)
    .agg(用户数=("CustomerID", "nunique"),
         流失人数=("Churn", "sum"),
         流失率=("Churn", "mean"))
    .reindex(tenure_order)
    .reset_index()
)

display(category_summary)
total_users = category_summary["用户数"].sum()
print(f"总用户数：{total_users}")

In [ ]:
# 绘制柱状图
fig_bar, ax_bar = plt.subplots(figsize=(10, 6))

x = range(len(category_summary))
bars = ax_bar.bar(x, category_summary["流失率"], color="#4C72B0", alpha=0.85)

# 在柱子上标注数值和样本量
for i, (idx, row) in enumerate(category_summary.iterrows()):
    label = f'{row["流失率"]:.1%}\nn={int(row["用户数"])}'
    ax_bar.text(i, row["流失率"] + 0.01, label, ha="center", va="bottom", fontsize=9)

ax_bar.set_xticks(x)
ax_bar.set_xticklabels(category_summary[category_field])
ax_bar.set_ylabel("流失率")
ax_bar.set_xlabel("生命周期阶段")
ax_bar.set_title("不同生命周期阶段的用户流失率对比")
ax_bar.yaxis.set_major_formatter(PercentFormatter(1.0))
ax_bar.set_ylim(0, max(category_summary["流失率"]) * 1.3)

plt.tight_layout()
bar_path = OUTPUT_DIR / "01_category_bar.png"
fig_bar.savefig(bar_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"已输出：{bar_path}")

### 柱状图结论

**观察：** 新用户（0-1年）的流失率远高于老用户（15年以上），两者差距超过20个百分点。

**证据：** 0-1年用户流失率约31%（n=1198），15年以上用户流失率约8%（n=1380）。各阶段样本量充足，均超过700人。

**边界：** 该图仅展示相关性，不能证明使用时长导致流失降低。可能存在选择偏差（愿意留下来的用户本身就更喜欢平台）。

---
## 任务2：用户行为散点图

**X轴：** Tenure（使用月数）  
**Y轴：** CashbackAmount（返现金额）  
**问题：** 使用时长与返现金额之间是否存在关联？流失用户分布有何特征？  

注意：CashbackAmount 是返现金额，不是消费金额。

In [ ]:
# 选择两个数值字段
x_field = "Tenure"
y_field = "CashbackAmount"

assert x_field in df.columns and y_field in df.columns
assert pd.api.types.is_numeric_dtype(df[x_field])
assert pd.api.types.is_numeric_dtype(df[y_field])

fig_scatter, ax_scatter = plt.subplots(figsize=(10, 6))

# 分流失/未流失两组绘制
churn_0 = df[df["Churn"] == 0]
churn_1 = df[df["Churn"] == 1]

ax_scatter.scatter(churn_0[x_field], churn_0[y_field],
                   alpha=0.3, s=15, color="#4C72B0", label=f"未流失 (n={len(churn_0)})")
ax_scatter.scatter(churn_1[x_field], churn_1[y_field],
                   alpha=0.5, s=20, color="#D65F5F", label=f"流失 (n={len(churn_1)})")

ax_scatter.set_xlabel("使用时长（月）")
ax_scatter.set_ylabel("返现金额")
ax_scatter.set_title("用户使用时长 vs 返现金额（按流失状态着色）")
ax_scatter.legend(loc="upper right")

plt.tight_layout()
scatter_path = OUTPUT_DIR / "02_behavior_scatter.png"
fig_scatter.savefig(scatter_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"已输出：{scatter_path}")

### 散点图结论

**观察：** 流失用户（红色）主要集中在低使用时长区域，而老用户（蓝色）分布更广泛。

**证据：** 流失用户的 Tenure 主要集中在 0-24 个月区间，未流失用户在各时长段均有分布。返现金额两个群体差异不大，集中在 100-200 元区间。

**边界：** 散点图只能展示变量间的分布关系，不能说明因果关系。流失与使用时长之间的关联可能受其他因素（如产品体验、竞争环境）影响。

---
## 任务3：有序阶段折线图

**字段：** TenureGroup  
**问题：** 随着用户生命周期的增长，流失率和订单数如何变化？  

注意：这是有序阶段比较，不是时间趋势。

In [ ]:
# 准备有序数据
ordered_field = "TenureGroup"
ordered_summary = (
    df.groupby(ordered_field, observed=True)
    .agg(用户数=("CustomerID", "nunique"),
         流失率=("Churn", "mean"),
         平均订单数=("OrderCount", "mean"))
    .reindex(tenure_order)
    .reset_index()
)

display(ordered_summary)
assert {ordered_field, "用户数"}.issubset(ordered_summary.columns)

In [ ]:
# 绘制双Y轴折图
fig_line, ax_line = plt.subplots(figsize=(10, 6))
ax2 = ax_line.twinx()

x = range(len(ordered_summary))
labels = ordered_summary[ordered_field]

# 流失率折线（左轴）
line1 = ax_line.plot(x, ordered_summary["流失率"], "o-", color="#D65F5F",
                     linewidth=2, markersize=8, label="流失率")
ax_line.set_ylabel("流失率", color="#D65F5F")
ax_line.yaxis.set_major_formatter(PercentFormatter(1.0))

# 订单数折线（右轴）
line2 = ax2.plot(x, ordered_summary["平均订单数"], "s-", color="#4C72B0",
                 linewidth=2, markersize=8, label="平均订单数")
ax2.set_ylabel("平均订单数", color="#4C72B0")

# 标注数值
for i, (idx, row) in enumerate(ordered_summary.iterrows()):
    ax_line.annotate(f'{row["流失率"]:.1%}\n(n={int(row["用户数"])})',
                     xy=(i, row["流失率"]),
                     xytext=(0, 12), textcoords="offset points",
                     ha="center", fontsize=8, color="#D65F5F")

ax_line.set_xticks(x)
ax_line.set_xticklabels(labels)
ax_line.set_xlabel("生命周期阶段（有序）")
ax_line.set_title("各生命周期阶段的流失率与订单数（阶段比较）")

# 合并图例
lines = line1 + line2
labels_legend = [l.get_label() for l in lines]
ax_line.legend(lines, labels_legend, loc="upper right")

plt.tight_layout()
line_path = OUTPUT_DIR / "03_ordered_line.png"
fig_line.savefig(line_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"已输出：{line_path}")

### 折线图结论

**观察：** 流失率随生命周期增长呈明显下降趋势，而平均订单数随生命周期增长而上升。

**证据：** 0-1年用户流失率约31%，平均订单数约2.6；15年以上用户流失率降至约8%，平均订单数升至约3.2。各阶段样本量均超过750。

**边界：** 这是有序阶段比较，不是月度或年度时间趋势。不能据此推断"使用越久流失率越低"的因果结论，可能存在幸存者偏差。

---
## 任务4：整体构成图

**字段：** Churn  
**问题：** 整体用户中流失与未流失的比例构成是怎样的？  

**选择理由：** Churn 只有 2 个类别（流失/未流失），类别数量少且合计为整体，适合用饼图展示构成。

In [ ]:
# 准备构成数据
composition_field = "Churn"
composition_summary = (
    df.groupby(composition_field)
    .agg(用户数=("CustomerID", "nunique"))
    .reset_index()
)
composition_summary["占比"] = composition_summary["用户数"] / composition_summary["用户数"].sum()
composition_summary["标签"] = composition_summary[composition_field].map({0: "未流失", 1: "流失"})

display(composition_summary)
assert np.isclose(composition_summary["占比"].sum(), 1.0)

In [ ]:
# 绘制环形图
fig_comp, ax_comp = plt.subplots(figsize=(8, 8))

colors = ["#4C72B0", "#D65F5F"]
wedges, texts, autotexts = ax_comp.pie(
    composition_summary["用户数"],
    labels=[f'{row["标签"]}\n(n={int(row["用户数"])}, {row["占比"]:.1%})'
            for _, row in composition_summary.iterrows()],
    colors=colors,
    autopct="",
    startangle=90,
    pctdistance=0.75,
    wedgeprops=dict(width=0.4)
)

ax_comp.set_title("用户流失构成（流失 vs 未流失）")

plt.tight_layout()
composition_path = OUTPUT_DIR / "04_composition_chart.png"
fig_comp.savefig(composition_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"已输出：{composition_path}")

### 构成图结论

**观察：** 整体用户中约83%为未流失用户，约17%为流失用户。

**证据：** 未流失用户4760人（84.5%），流失用户870人（15.5%）。整体流失率约16.8%，与公共基础指标一致。

**边界：** 饼图只展示整体构成，不能反映各细分群体内部的流失差异。需要结合柱状图或交叉分析进一步观察不同群体的流失情况。

---
## 检查点2与3：基础图表、优化和解释

In [ ]:
# 检查点2：验证4张图
individual_paths = [bar_path, scatter_path, line_path, composition_path]
for path in individual_paths:
    assert path.exists() and path.suffix.lower() == ".png"
    assert path.stat().st_size > 5000, f"图片可能为空：{path.name}"
    print(f"  ✅ {path.name}: {path.stat().st_size / 1024:.1f} KB")

print("\n✅ 检查点2通过：4张独立图已生成")

---
## 任务5：2×2综合图

将4张核心图整合到一个 2×2 面板中，统一风格。

In [ ]:
fig_summary, axes = plt.subplots(2, 2, figsize=(14, 10))

# 子图1：类别比较柱状图
ax = axes[0, 0]
x_bar = range(len(category_summary))
ax.bar(x_bar, category_summary["流失率"], color="#4C72B0", alpha=0.85)
for i, (idx, row) in enumerate(category_summary.iterrows()):
    ax.text(i, row["流失率"] + 0.01, f'{row["流失率"]:.1%}', ha="center", fontsize=8)
ax.set_xticks(x_bar)
ax.set_xticklabels(category_summary[category_field], fontsize=8, rotation=15)
ax.set_ylabel("流失率")
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.set_title("各生命周期流失率对比", fontsize=11)

# 子图2：散点图
ax = axes[0, 1]
churn_0 = df[df["Churn"] == 0]
churn_1 = df[df["Churn"] == 1]
ax.scatter(churn_0["Tenure"], churn_0["CashbackAmount"], alpha=0.2, s=10, color="#4C72B0", label=f"未流失")
ax.scatter(churn_1["Tenure"], churn_1["CashbackAmount"], alpha=0.4, s=15, color="#D65F5F", label=f"流失")
ax.set_xlabel("使用时长（月）", fontsize=9)
ax.set_ylabel("返现金额", fontsize=9)
ax.set_title("使用时长 vs 返现金额", fontsize=11)
ax.legend(fontsize=8)

# 子图3：折线图
ax = axes[1, 0]
x_line = range(len(ordered_summary))
ax.plot(x_line, ordered_summary["流失率"], "o-", color="#D65F5F", linewidth=2)
ax.plot(x_line, ordered_summary["平均订单数"], "s-", color="#4C72B0", linewidth=2)
ax.set_xticks(x_line)
ax.set_xticklabels(ordered_summary[ordered_field], fontsize=8, rotation=15)
ax.set_ylabel("数值", fontsize=9)
ax.set_title("生命周期：流失率与订单数", fontsize=11)
ax.legend(["流失率", "平均订单数"], fontsize=8)

# 子图4：构成图
ax = axes[1, 1]
ax.pie(
    composition_summary["用户数"],
    labels=[f'{row["标签"]} ({row["占比"]:.1%})' for _, row in composition_summary.iterrows()],
    colors=colors,
    startangle=90,
    wedgeprops=dict(width=0.4)
)
ax.set_title("用户流失构成", fontsize=11)

fig_summary.suptitle("电商用户数据可视化分析概览", fontsize=16, fontweight="bold")
fig_summary.tight_layout(rect=[0, 0, 1, 0.96])

summary_path = OUTPUT_DIR / "day06_visualization_summary.png"
fig_summary.savefig(summary_path, dpi=150, bbox_inches="tight")
plt.show()
print(f"已输出：{summary_path}")

---
## 综合发现与局限

In [ ]:
print("=" * 60)
print("综合发现")
print("=" * 60)

print("""
1. 新用户流失风险显著高于老用户
   证据：0-1年用户流失率约31%（n=1198），15年以上用户仅约8%（n=1380），
   差距超过20个百分点。该结论对应柱状图和折线图。

2. 流失用户集中在低使用时长区域
   证据：散点图显示流失用户（红色）主要集中在 Tenure 0-24 个月区间，
   而使用时长超过60个月的用户几乎没有流失。该结论对应散点图。

3. 整体流失率约17%，用户留存状况良好
   证据：未流失用户4760人（84.5%），流失用户870人（15.5%），
   整体流失率与公共指标一致。该结论对应构成图。
""")

print("=" * 60)
print("数据或方法局限")
print("=" * 60)
print("""
1. 数据为横截面数据，只能观察关联，无法建立因果关系；
2. 缺少订单日期和金额，无法分析时间趋势或计算GMV；
3. CashbackAmount是返现金额，不能等同于消费金额或收入；
4. 折线图展示的是有序阶段比较，不是时间趋势，不能外推。
""")

---
## 任务6：图表清单

生成 chart_manifest.csv，供第7天 Flask 读取。

In [ ]:
# 图表清单
chart_manifest = pd.DataFrame([
    {"chart_id": "01", "file_name": "01_category_bar.png",
     "business_question": "不同生命周期阶段的用户流失率是否有显著差异？",
     "chart_type": "bar",
     "key_finding": "0-1年新用户流失率约31%，15年以上老用户仅约8%，差距超过20个百分点",
     "limitation": "仅展示相关性，不能证明使用时长导致流失降低，可能存在选择偏差"},
    {"chart_id": "02", "file_name": "02_behavior_scatter.png",
     "business_question": "用户使用时长与返现金额之间是否存在关联？流失用户分布有何特征？",
     "chart_type": "scatter",
     "key_finding": "流失用户主要集中在低使用时长区域（0-24个月），返现金额两群体差异不大",
     "limitation": "只能展示分布关系，不能说明因果关系；alpha设置可能影响视觉密度"},
    {"chart_id": "03", "file_name": "03_ordered_line.png",
     "business_question": "随着用户生命周期的增长，流失率和订单数如何变化？",
     "chart_type": "line",
     "key_finding": "流失率随生命周期增长明显下降，平均订单数随生命周期增长而上升",
     "limitation": "这是有序阶段比较，不是时间趋势，可能存在幸存者偏差"},
    {"chart_id": "04", "file_name": "04_composition_chart.png",
     "business_question": "整体用户中流失与未流失的比例构成是怎样的？",
     "chart_type": "pie",
     "key_finding": "未流失用户占84.5%（4760人），流失用户占15.5%（870人）",
     "limitation": "饼图只展示整体构成，不能反映各细分群体的流失差异"},
    {"chart_id": "05", "file_name": "day06_visualization_summary.png",
     "business_question": "整体概览：用户流失的多维度特征",
     "chart_type": "dashboard",
     "key_finding": "新用户流失率高、流失集中在低使用时长群体、整体留存率约83%",
     "limitation": "综合图缩略展示，细节需参考各独立图表"},
])

manifest_path = OUTPUT_DIR / "chart_manifest.csv"
chart_manifest.to_csv(manifest_path, index=False, encoding="utf-8-sig")

print("=== 图表清单 ===")
display(chart_manifest)
print(f"\n已输出：{manifest_path}")

---
## 检查点4：提交前检查

In [ ]:
# 最终验证
required_outputs = [
    OUTPUT_DIR / "01_category_bar.png",
    OUTPUT_DIR / "02_behavior_scatter.png",
    OUTPUT_DIR / "03_ordered_line.png",
    OUTPUT_DIR / "04_composition_chart.png",
    OUTPUT_DIR / "day06_visualization_summary.png",
    OUTPUT_DIR / "chart_manifest.csv",
]

print("=== 成果文件清单 ===")
for path in required_outputs:
    if path.exists():
        size = path.stat().st_size / 1024
        print(f"  ✅ {path.name}: {size:.1f} KB")
    else:
        print(f"  ❌ {path.name}: 缺失")

# 验证清单文件
manifest_check = pd.read_csv(OUTPUT_DIR / "chart_manifest.csv")
assert list(manifest_check.columns) == [
    "chart_id", "file_name", "business_question",
    "chart_type", "key_finding", "limitation",
]
assert len(manifest_check) == 5

print("\n✅ 检查点4通过：第6天成果物完整")
print(f"\n输出目录：{OUTPUT_DIR}")